In [2]:
# Steo-1: Import Libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

In [4]:
#Load Datase

df = pd.read_csv('C:/Users/hp/Desktop/Assignments/Ass-5/raw_loan_dataset.csv')


In [5]:

print("\n=== before HEAD ===")
print(df.head())


=== before HEAD ===
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  


In [6]:
print("\n=== before INFO ===")
print(df.info())


=== before INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB
None


In [7]:
#check mission value

df.isnull().sum()


Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64

In [8]:
# Steo-2: Clean currency formatting

df["Income"] = df["Income"].replace(r"[$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[$,]", "", regex=True).astype(float)


In [9]:
# Steo-3: Normalize categorical typos Before

def normalize_cat(val):
    val = str(val).lower().strip()
    if val in ['yes', 'y', 'approved']: return 'Yes'
    if val in ['no', 'n', 'rejected']: return 'No'
    return np.nan

for col in ['HasCollateral', 'PreviousDefaults', 'Approved']:
    df[col] = df[col].apply(normalize_cat)
    print(f"{col} counts:\n", df[col].value_counts())
print(df.head())

HasCollateral counts:
 HasCollateral
No     51
Yes    49
Name: count, dtype: int64
PreviousDefaults counts:
 PreviousDefaults
No     85
Yes    14
Name: count, dtype: int64
Approved counts:
 Approved
Yes    68
No     35
Name: count, dtype: int64
     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  


In [10]:
# Steo-4: handding missing values

df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])


In [12]:
# Steo-5: check duplicates and Remove

df.duplicated().sum()


np.int64(3)

In [13]:
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


In [14]:
# Steo-6: handding outlier with methond capping

def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

In [16]:
# Step-7: Label Encoding Categorical Features

map_dict = {'Yes': 1, 'No': 0}
for col in ['Approved', 'HasCollateral', 'PreviousDefaults']:
    df[col] = df[col].map(map_dict)

In [17]:
df.shape

(100, 7)

In [18]:
df.dtypes

Income              float64
CreditScore         float64
EmploymentYears     float64
LoanAmount          float64
HasCollateral         int64
PreviousDefaults      int64
Approved              int64
dtype: object

In [19]:
# Step-8: Class balance check 

class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")



Class balance OK for baseline Accuracy (both classes well represented).


In [20]:
# Step-9: Feature engineering

df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

In [21]:
scaler = RobustScaler()
num_cols = ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount', 'DebtToIncome', 'IncomePerYearEmployed']
df[num_cols] = scaler.fit_transform(df[num_cols])

#RobustScaler: Why I Used It:
    
For this loan approval data set, I went with RobustScaler because it is built for datasets with outliers.
Financial features like Income and LoanAmount often have extreme values which can adversely affect other scaling methods
like StandardScaler and MinMaxScaler. RobustScaler scales the data according to median and the Interquartile Range (IQR)
instead of mean and standard deviation that makes it robust to outliers Thus, 
RobustScaler is a good choice for loan prediction as it maintains the distribution of most applicants while reducing the effect of outliers.
This dataset has financial and credit related information which could have had skewed distributions and outliers. 
Hence we could use RobustScaler for more reliable feature scaling.
    

In [23]:
# Step-10:Final snapshot

print("\n=== FINAL HEAD ===")
print(df.head())


=== FINAL HEAD ===
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445244               8.274536  
1                 0         1     -0.912749               0.153994  
2                 0         1      0.280113              -0.126321  
3                 0         1     -0.315175              -0.669033  
4                 0         1     -0.284688              -0.284975  


In [24]:
print("\n=== FINAL INFO ===")
print(df.info())


=== FINAL INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.8 KB
None


In [26]:
print("\n=== FINAL MISSING VALUES CHECK ===")
print(df.isnull().sum())


=== FINAL MISSING VALUES CHECK ===
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


In [28]:
OUT_PATH = 'Processed_loan_data.csv'
df.to_csv(OUT_PATH, index=False)